In [ ]:
import pandas as pd

# Vamos carregar o arquivo principal (ajuste o nome se necessário)
# Se você criou o notebook dentro da pasta challenge_sauter, o caminho é direto:
df = pd.read_csv('fipe_2022.csv')

# Exibir as primeiras linhas para entender a estrutura
print("Primeiras linhas do dataset:")
display(df.head())

# Verificar os tipos de dados (importante para o motor matemático)
print("\nInformações técnicas das colunas:")
df.info()

Primeiras linhas do dataset:


,year_of_reference,month_of_reference,fipe_code,authentication,brand,model,fuel,gear,engine_size,year_model,avg_price_brl,age_years
0,2022,January,038001-6,vwmrywl5qs,Acura,NSX 3.0,Gasoline,manual,3.0,1995,43779.0,28
1,2022,January,038001-6,t9mt723qhz,Acura,NSX 3.0,Gasoline,manual,3.0,1994,42244.0,29
2,2022,January,038001-6,tr5wv4z21g,Acura,NSX 3.0,Gasoline,manual,3.0,1993,40841.0,30
3,2022,January,038001-6,s2xxsjz3mt,Acura,NSX 3.0,Gasoline,manual,3.0,1992,39028.0,31
4,2022,January,038001-6,rtm9gj7zk8,Acura,NSX 3.0,Gasoline,manual,3.0,1991,35678.0,32



Informações técnicas das colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290275 entries, 0 to 290274
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   year_of_reference   290275 non-null  int64  
 1   month_of_reference  290275 non-null  object 
 2   fipe_code           290275 non-null  object 
 3   authentication      290275 non-null  object 
 4   brand               290275 non-null  object 
 5   model               290275 non-null  object 
 6   fuel                290275 non-null  object 
 7   gear                290275 non-null  object 
 8   engine_size         290275 non-null  float64
 9   year_model          290275 non-null  int64  
 10  avg_price_brl       290275 non-null  float64
 11  age_years           290275 non-null  int64  
dtypes: float64(2), int64(3), object(7)
memory usage: 26.6+ MB


In [ ]:
# 1. Filtrar para apenas um mês de referência (evita duplicatas e acelera o código)
# Vamos pegar o mês mais recente disponível no seu print (Janeiro de 2022)
df_limpo = df[df['month_of_reference'] == 'January'].copy()

# 2. Selecionar features incluindo 'age_years' como proxy para quilometragem
# Descartamos fipe_code, authentication, year_of_reference e month_of_reference
features = ['brand', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl', 'age_years']
df_filtered = df_limpo[features].copy()

print(f"Dataset reduzido de {len(df)} para {len(df_filtered)} linhas (sem duplicatas mensais).")

Dataset reduzido de 290275 para 24031 linhas (sem duplicatas mensais).


In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Selecionar apenas as colunas úteis para a recomendação
# Vamos descartar códigos internos e datas de referência que não definem o "perfil" do carro
features = ['brand', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl']
df_filtered = df[features].copy()

# 2. Definir quais colunas são texto (categóricas) e quais são números
categorical_features = ['fuel', 'gear']
numerical_features = ['engine_size', 'year_model', 'avg_price_brl']

# 3. Criar o transformador: 
# - OneHotEncoder transforma "Manual/Automatic" em 0 e 1
# - StandardScaler coloca Preço e Motor na mesma escala (Média 0, Desvio 1)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# 4. Aplicar a transformação e gerar a matriz matemática
matrix_transformed = preprocessor.fit_transform(df_filtered)

print("Matriz preparada para o motor de recomendação!")
print(f"Formato da matriz: {matrix_transformed.shape}")

Matriz preparada para o motor de recomendação!
Formato da matriz: (290275, 8)


In [ ]:
def recomendar_concorrentes(id_carro_escolhido, top_n=5, margem_preco=0.20):
    # 1. Pegar o vetor do carro e transformar em 2D (Correção do Erro)
    # O .reshape(1, -1) transforma o vetor de 1D para 2D como o sklearn exige
    carro_vetor = matrix_transformed[id_carro_escolhido].reshape(1, -1)
    
    # 2. Calcular a similaridade contra todos os outros
    sim_scores = cosine_similarity(carro_vetor, matrix_transformed).flatten()
    
    # 3. Adicionar os scores ao dataframe para filtrar
    df_analise = df_filtered.copy()
    df_analise['similarity'] = sim_scores
    
    # Dados do carro de referência
    carro_ref = df_filtered.iloc[id_carro_escolhido]
    preco_ref = carro_ref['avg_price_brl']
    marca_ref = carro_ref['brand']
    
    # 4. Aplicar Regras de Negócio
    filtro_marcas = df_analise['brand'] != marca_ref
    preco_min, preco_max = preco_ref * 0.75, preco_ref * 1.25 # Margem de segurança
    filtro_preco = df_analise['avg_price_brl'].between(preco_min, preco_max)
    
    # 5. Filtrar e ordenar
    recomendacoes = df_analise[filtro_marcas & filtro_preco].sort_values(by='similarity', ascending=False)
    
    return carro_ref, recomendacoes.head(top_n)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Filtro de Janeiro e inclusao da coluna 'model'
df_janeiro = df[df['month_of_reference'] == 'January'].copy()

# Lista de colunas necessarias (incluindo 'model' para a interface)
features_list = ['brand', 'model', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl', 'age_years']
df_filtered = df_janeiro[features_list].copy().reset_index(drop=True)

# 2. Re-treinar a matriz para garantir o tamanho correto (24031 linhas)
num_features = ['engine_size', 'year_model', 'avg_price_brl', 'age_years']
cat_features = ['fuel', 'gear']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(), cat_features)
])

matrix_transformed = preprocessor.fit_transform(df_filtered)

print("Dados preparados com sucesso. Coluna 'model' incluida.")

Dados preparados com sucesso. Coluna 'model' incluida.


In [ ]:
# 1. Criar a coluna de busca caso ela nao exista
if 'search_name' not in df_filtered.columns:
    df_filtered['search_name'] = df_filtered['brand'] + " " + df_filtered['model'] + " (" + df_filtered['year_model'].astype(str) + ")"

# 2. Campo de preenchimento (O Python abrira uma caixa de texto para voce digitar)
termo_usuario = input("Digite a marca ou modelo do carro que voce esta vendo: ")

# 3. Buscar o veiculo no dataset
matches = df_filtered[df_filtered['search_name'].str.contains(termo_usuario, case=False)]

if matches.empty:
    print(f"Nenhum veiculo encontrado para: '{termo_usuario}'. Tente digitar apenas a marca ou o modelo principal.")
else:
    # Selecionamos o primeiro resultado encontrado na busca
    id_escolhido = matches.index[0]
    
    # 4. Chamar a funcao de recomendacao (que ja criamos anteriormente)
    alvo, sugestoes = recomendar_concorrentes(id_escolhido)
    
    # 5. Exibicao dos resultados
    print("\nSISTEMA DE RECOMENDACAO - SAUTER DIGITAL")
    print("-" * 90)
    print(f"VOCE ESTA VENDO: {alvo['brand']} {alvo['model']} | Motor: {alvo['engine_size']} | Preco: R${alvo['avg_price_brl']:,.2f}")
    print("-" * 90)
    print("OUTROS QUE VOCE PODE GOSTAR (MARCAS CONCORRENTES):")
    
    if sugestoes.empty:
        print("Nenhuma alternativa encontrada com os filtros de preco e marca atuais.")
    else:
        # Exibe a tabela final
        display(sugestoes[['brand', 'model', 'engine_size', 'year_model', 'avg_price_brl', 'score_similaridade']])